> The monitoring data and building-specific configuration are confidential and are not included. See `README.md` for execution requirements. Clear any generated outputs before sharing an executed notebook.


In [ ]:
# Private workspace and configuration
from copy import deepcopy as _privacy_deepcopy
from functools import lru_cache as _privacy_lru_cache
import json
import os
from pathlib import Path


def _private_config_path(variable):
    value = os.environ.get(variable)
    if not value:
        raise RuntimeError(f"Set {variable} before running this notebook; see README.md.")
    return Path(value).expanduser().resolve()


def private_workspace():
    """Return the configured private input/output directory."""
    path = _private_config_path("HVAC_PRIVATE_WORKSPACE")
    if not path.is_dir():
        raise FileNotFoundError("The configured private workspace does not exist.")
    return path


def _decode_private_values(value):
    if isinstance(value, list):
        return [_decode_private_values(item) for item in value]
    if isinstance(value, dict):
        if set(value) == {"__tuple_key_dict__"}:
            return {tuple(key): _decode_private_values(item)
                    for key, item in value["__tuple_key_dict__"]}
        return {key: _decode_private_values(item) for key, item in value.items()}
    return value


@_privacy_lru_cache(maxsize=1)
def _load_private_values():
    with _private_config_path("HVAC_PRIVATE_SETTINGS").open(encoding="utf-8") as handle:
        values = json.load(handle).get("private_values")
    if not isinstance(values, dict):
        raise ValueError("Private settings must contain a private_values object.")
    return _decode_private_values(values)


def private_value(key):
    """Read a private setting without embedding its value in the public source."""
    values = _load_private_values()
    if key not in values or values[key] is None:
        raise KeyError(f"Required private setting is missing: {key!r}.")
    return _privacy_deepcopy(values[key])


# Notebook 02: Temperature models and duration-aware neural training

This notebook fits room- and operating-mode-specific Extra Trees temperature models and one neural network shared across rooms and both modes. The tree models predict temperature change at a requested horizon. The neural network predicts the directed rate \(v=s\,dT/dt\), where \(s=+1\) for heating and \(-1\) for cooling; physical rates are integrated to reconstruct temperatures.

Two datasets provide supervision: long-format future temperature/rate targets and dedicated remaining-duration prediction rows, called anchors in the code. The neural loss combines rate, temperature and duration errors. Only the training duration term is weighted by the trajectory-regularity score \(q\). The network has no explicit room identifier or separate duration output.

All models use Notebook 01's chronological split. Imputation, scaling and target normalization use training data. The neural checkpoint, learning-rate schedule and early stopping use validation duration error; test data are reserved for evaluation. Offline grid reconstruction is inherited from Notebook 01 and is separate from these training-only fitted transformations.

The training loop supports checkpoint recovery and mixed-precision network operations. Integration and duration calculations use 32-bit floating-point arithmetic. Private paths, data and configuration must be supplied as described in `README.md`.


In [ ]:

# Imports, reproducibility, GPU settings, persistent run paths

import os, re, json, math, random, warnings, shutil, hashlib, time
from pathlib import Path
from itertools import cycle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import joblib

try:
    from IPython.display import display
except Exception:
    def display(x): print(x)

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset
except Exception as e:
    raise RuntimeError('PyTorch is required for Notebook 02.') from e

try:
    from google.colab import drive  # type: ignore
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except Exception:
    pass

def safe_torch_load(path,map_location=None):
    try:
        return torch.load(path,map_location=map_location,weights_only=False)
    except TypeError:  # compatibility with older PyTorch
        return torch.load(path,map_location=map_location)

warnings.filterwarnings('default')
RANDOM_SEED=42
STRICT_DETERMINISM=False  # Random generators are seeded; strict deterministic kernels are optional.

def reset_seeds(seed=RANDOM_SEED):
    np.random.seed(seed); random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

reset_seeds()
if STRICT_DETERMINISM:
    torch.use_deterministic_algorithms(True,warn_only=True)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic=True
        torch.backends.cudnn.benchmark=False
else:
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.benchmark=True

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type=='cuda':
    torch.set_float32_matmul_precision('high')
    if hasattr(torch.backends,'cuda'):
        torch.backends.cuda.matmul.allow_tf32=True
    props=torch.cuda.get_device_properties(0)
    GPU_MEMORY_GB=props.total_memory/1024**3
    print('GPU:',props.name,f'({GPU_MEMORY_GB:.1f} GB)')
else:
    GPU_MEMORY_GB=0.0
print('DEVICE:',DEVICE)

# AMP accelerates the MLP. Physical integration/crossing functions explicitly cast to
# FP32.
USE_AMP=bool(DEVICE.type=='cuda')
AMP_DTYPE=torch.float16
GRAD_SCALER=torch.cuda.amp.GradScaler(enabled=USE_AMP)

BASE_DIR=Path(str(private_workspace()))
DATASET_FOLDER='full_rampup_5min_feature_dataset_UNIFIED_DERIVATIVE_DURATION_AWARE'
DATA_DIR=BASE_DIR/DATASET_FOLDER
SPLIT_MAP_PATH=DATA_DIR/'shared_event_split_map.csv'
SPLIT_SIGNATURE_PATH=DATA_DIR/'shared_event_split_signature.json'

# Persistent name is intentional: rerunning after a Colab disconnect can resume.
TRAINING_ROOT=BASE_DIR/'training_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL'
RUN_NAME='paper_final_q_weighted'
OUT_ROOT=TRAINING_ROOT/RUN_NAME
MODEL_DIR=OUT_ROOT/'models'; TABLE_DIR=OUT_ROOT/'tables'; FIG_DIR=OUT_ROOT/'figures'; PRED_DIR=OUT_ROOT/'predictions'; AUDIT_DIR=OUT_ROOT/'audit'; CHECKPOINT_DIR=OUT_ROOT/'checkpoints'
for p in [MODEL_DIR,TABLE_DIR,FIG_DIR,PRED_DIR,AUDIT_DIR,CHECKPOINT_DIR]: p.mkdir(parents=True,exist_ok=True)

# Set RESET_RUN=True only when deliberately starting a completely new optimization.
RESET_RUN=False
RESUME_TRAINING=True
FORCE_CONTINUE_AFTER_EARLY_STOP=False
if RESET_RUN and OUT_ROOT.exists():
    shutil.rmtree(OUT_ROOT)
    for p in [MODEL_DIR,TABLE_DIR,FIG_DIR,PRED_DIR,AUDIT_DIR,CHECKPOINT_DIR]: p.mkdir(parents=True,exist_ok=True)

PHASES=['heating','cooling']
STEP_MIN=5.0
END_BAND_C=0.4
DURATION_MAX_HORIZON_BY_PHASE={'heating':185.0,'cooling':65.0}  # extra 5 min safely brackets event_start_asof source-time gaps
PAPER_TEMPERATURE_XMAX_BY_PHASE={'heating':180.0,'cooling':60.0}
MIN_ET_TRAIN_ROWS_PER_ROOM=50
ET_PARAMS=dict(n_estimators=600,min_samples_leaf=2,max_features=0.8,random_state=RANDOM_SEED,n_jobs=-1)

NN_HIDDEN_DIMS=[128,64,32]
NN_DROPOUT=0.10
NN_LR=1e-3
NN_WEIGHT_DECAY=1e-4
NN_BATCH_TRAJECTORIES_PER_PHASE=256 if GPU_MEMORY_GB>=10 else 128
NN_BATCH_DURATION_ANCHORS_PER_PHASE=256 if GPU_MEMORY_GB>=10 else 128
NN_MAX_EPOCHS=500
NN_VALIDATION_EVERY=2
NN_EARLY_STOP_PATIENCE_VALIDATIONS=10
NN_SCHEDULER_PATIENCE_VALIDATIONS=2
NN_SCHEDULER_FACTOR=0.5
NN_MIN_LR=1e-6
NN_MIN_DELTA=1e-5
SELECTED_SCORE_MODE='score_weighted'

SOFT_TEMP_BETA_C=0.04
SOFT_DIRECTION_BETA_FLOOR_C=0.005

print('DATA_DIR:',DATA_DIR)
print('OUT_ROOT:',OUT_ROOT)
print('Trajectory batch / phase:',NN_BATCH_TRAJECTORIES_PER_PHASE)
print('Duration-anchor batch / phase:',NN_BATCH_DURATION_ANCHORS_PER_PHASE)
print('AMP:',USE_AMP)


In [ ]:

# Load long-format temperature trajectories + saved duration anchors + shared split

def norm_event_id(x):
    if pd.isna(x): raise RuntimeError('Missing event_id')
    s = str(x).strip()
    if re.fullmatch(r'[-+]?\d+\.0+', s): s = str(int(float(s)))
    return s

def load_phase_long(phase):
    phase_dir = DATA_DIR/phase
    main = phase_dir/f'{phase}_full_rampup_5min_samples_LONG.csv'
    by_room = sorted((phase_dir/'by_room').glob(f'*_{phase}_full_rampup_5min_LONG.csv'))
    frames=[]
    if main.exists(): frames.append(pd.read_csv(main))
    frames += [pd.read_csv(p) for p in by_room]
    if not frames: raise FileNotFoundError(f'No LONG data found for {phase}')
    d=pd.concat(frames,ignore_index=True,sort=False)
    d['phase']=phase
    for c in ['timestamp','target_timestamp','rampup_start_time','rampup_end_time','rampup_end_grid_time','extraction_end_time']:
        if c in d.columns: d[c]=pd.to_datetime(d[c],errors='coerce')
    d['event_id']=d['event_id'].map(norm_event_id)
    d['room']=d['room'].astype(str).str.strip()
    d['shared_event_uid']=d['phase']+'|'+d['room']+'|'+d['event_id']
    key=['phase','room','event_id','timestamp','target_timestamp','horizon_min']
    return d.drop_duplicates(subset=[c for c in key if c in d.columns],keep='first').reset_index(drop=True)

split_map=pd.read_csv(SPLIT_MAP_PATH)
split_map['event_id']=split_map['event_id'].map(norm_event_id)
split_map['room']=split_map['room'].astype(str).str.strip()
split_map['shared_event_uid']=split_map['phase'].astype(str)+'|'+split_map['room']+'|'+split_map['event_id']
split_map=split_map[['shared_event_uid','split','split_block']].drop_duplicates()

# long-format temperature/rate supervision.
phase_frames=[]
for phase in PHASES:
    d=load_phase_long(phase)
    d=d.merge(split_map,on='shared_event_uid',how='left',validate='many_to_one')
    if d['split'].isna().any():
        bad=d[d['split'].isna()].copy(); bad.to_csv(AUDIT_DIR/f'{phase}_BAD_unmatched_LONG_split.csv',index=False)
        raise RuntimeError(f'{phase}: LONG rows do not match authoritative split map.')
    d['event_key']=d['shared_event_uid']
    d['target_context']=(d['target_context'] if 'target_context' in d.columns else pd.Series('inside_event',index=d.index)).astype(str).str.strip().str.lower()
    phase_frames.append(d)

DATA=pd.concat(phase_frames,ignore_index=True,sort=False)
DATA['_row_id']=np.arange(len(DATA),dtype=int)
DATA['use_for_temperature_training']=DATA['target_context'].isin(['inside_event','post_end_trend_continuation','post_end_buffer'])

required_long=['phase','room','event_key','split','timestamp','target_timestamp','horizon_min','Room_T','target_Room_T','thermal_mode_sign','Phase_Target_T_C','Directed_Distance_to_Target_C','Room_dTdt_now_bwd2_C_per_min','target_directed_rate_C_per_min']
missing=[c for c in required_long if c not in DATA.columns]
if missing: raise RuntimeError(f'Notebook 01 LONG columns missing: {missing}')
for c in ['horizon_min','Room_T','target_Room_T','thermal_mode_sign','Phase_Target_T_C','Directed_Distance_to_Target_C','Room_dTdt_now_bwd2_C_per_min','target_directed_rate_C_per_min']:
    DATA[c]=pd.to_numeric(DATA[c],errors='coerce')

# Dedicated duration rows define the duration supervision set independently of
# temperature-target rows.
anchor_frames=[]
for phase in PHASES:
    p=DATA_DIR/phase/f'{phase}_remaining_duration_anchors_5min.csv'
    if not p.exists(): raise FileNotFoundError(p)
    a=pd.read_csv(p); a['phase']=phase
    for c in ['timestamp','anchor_timestamp','feature_source_timestamp','rampup_start_time','rampup_end_time','rampup_end_grid_time']:
        if c in a.columns: a[c]=pd.to_datetime(a[c],errors='coerce')
    a['event_id']=a['event_id'].map(norm_event_id)
    a['room']=a['room'].astype(str).str.strip()
    a['shared_event_uid']=a['phase']+'|'+a['room']+'|'+a['event_id']
    a=a.merge(split_map,on='shared_event_uid',how='left',validate='many_to_one')
    if a['split'].isna().any():
        bad=a[a['split'].isna()].copy(); bad.to_csv(AUDIT_DIR/f'{phase}_BAD_unmatched_duration_split.csv',index=False)
        raise RuntimeError(f'{phase}: duration anchors do not match authoritative split map.')
    a['event_key']=a['shared_event_uid']
    a['true_remaining_duration_min']=pd.to_numeric(a.get('true_remaining_duration_min',a.get('remaining_duration_min')),errors='coerce')
    a['remaining_trajectory_quality_score']=pd.to_numeric(a['remaining_trajectory_quality_score'],errors='coerce').clip(0.0,1.0)
    a['thermal_mode_sign']=pd.to_numeric(a['thermal_mode_sign'],errors='coerce')
    a['Phase_Target_T_C']=pd.to_numeric(a['Phase_Target_T_C'],errors='coerce')
    a['Room_T']=pd.to_numeric(a['Room_T'],errors='coerce')
    a['asof_gap_min']=(a['anchor_timestamp']-a['feature_source_timestamp']).dt.total_seconds()/60.0
    a['asof_gap_min']=a['asof_gap_min'].clip(lower=0.0)
    kind=(a['anchor_kind'] if 'anchor_kind' in a.columns else pd.Series('anchor',index=a.index)).astype(str)
    a['anchor_kind']=kind
    a['anchor_key']=a['event_key']+'|'+a['anchor_timestamp'].astype(str)+'|'+kind
    a['Duration_Crossing_Threshold_C']=a['Phase_Target_T_C']-a['thermal_mode_sign']*END_BAND_C
    anchor_frames.append(a)

ANCHORS=pd.concat(anchor_frames,ignore_index=True,sort=False)
ANCHORS=ANCHORS.drop_duplicates('anchor_key',keep='first').reset_index(drop=True)
required_anchor=['phase','room','event_key','split','anchor_key','anchor_timestamp','feature_source_timestamp','Room_T','Phase_Target_T_C','thermal_mode_sign','true_remaining_duration_min','remaining_trajectory_quality_score','asof_gap_min']
missing=[c for c in required_anchor if c not in ANCHORS.columns]
if missing: raise RuntimeError(f'Notebook 01 duration-anchor columns missing: {missing}')

print('LONG rows:',len(DATA),'events:',DATA['event_key'].nunique(),'rooms:',DATA['room'].nunique())
display(DATA.groupby(['phase','split']).agg(rows=('_row_id','size'),events=('event_key','nunique'),rooms=('room','nunique')))
print('\nAUTHORITATIVE duration anchors:',len(ANCHORS),'events:',ANCHORS['event_key'].nunique())
display(ANCHORS.groupby(['phase','split']).agg(anchors=('anchor_key','nunique'),events=('event_key','nunique'),rooms=('room','nunique')))

# Audit that event-start anchors are present rather than silently lost by long-format
# horizon requirements.
if 'anchor_kind' in ANCHORS.columns:
    display(ANCHORS.groupby(['phase','split','anchor_kind']).size().rename('anchors').reset_index())

# Reference Consistency check — every model uses this exact split

def canonical_split_signature(split_frame):
    z=split_frame.copy()
    if 'shared_event_uid' not in z.columns:
        z['shared_event_uid']=z['phase'].astype(str)+'|'+z['room'].astype(str).str.strip()+'|'+z['event_id'].astype(str).str.strip()
    # Re-expand event UID only for a stable fingerprint compatible with Notebook 01.
    parts=z['shared_event_uid'].astype(str).str.split('|',n=2,expand=True)
    sig=pd.DataFrame({
        'phase':parts[0],
        'room':parts[1],
        'event_id':parts[2],
        'split':z['split'].astype(str),
        'split_block':z['split_block'].astype(str),
    }).drop_duplicates().sort_values(['phase','room','event_id']).reset_index(drop=True)
    canonical=sig.to_csv(index=False,lineterminator='\n')
    return hashlib.sha256(canonical.encode('utf-8')).hexdigest(),sig

SHARED_SPLIT_SHA256,SHARED_SPLIT_CANONICAL=canonical_split_signature(split_map)
if SPLIT_SIGNATURE_PATH.exists():
    with open(SPLIT_SIGNATURE_PATH) as f:
        expected_sig=json.load(f)
    if SHARED_SPLIT_SHA256!=expected_sig.get('sha256'):
        raise RuntimeError('Shared split SHA256 differs from Notebook 01. STOP: models would not use the same chronological split.')

# Verify that temperature and duration rows inherit the saved event split.
for label,frame in [('LONG',DATA),('DURATION_ANCHORS',ANCHORS)]:
    if frame['split'].isna().any(): raise RuntimeError(f'{label}: missing shared split assignments.')
    if not set(frame['split'].unique()).issubset({'train','valid','test'}): raise RuntimeError(f'{label}: unexpected split labels.')

split_audit=(
    SHARED_SPLIT_CANONICAL.groupby(['phase','split'],as_index=False)
    .size().rename(columns={'size':'events'})
)
split_audit.to_csv(AUDIT_DIR/'AUTHORITATIVE_shared_split_counts.csv',index=False)
with open(AUDIT_DIR/'AUTHORITATIVE_shared_split_signature.json','w') as f:
    json.dump({'sha256':SHARED_SPLIT_SHA256,'counts':split_audit.to_dict(orient='records')},f,indent=2)
print('Verified authoritative shared split SHA256:',SHARED_SPLIT_SHA256)
display(split_audit)


In [ ]:

# Select inputs without explicit room identifiers or raw temperature-history lags

DERIVATIVE_HISTORY_LAGS_MINUTES = list(range(5,65,5))
SOURCE_DERIVATIVE_HISTORY_LAGS_MINUTES = list(range(5,30,5))

COMMON_INPUTS = [
    'horizon_min',
    'Room_T', 'Room_V', 'Phase_Target_T_C', 'Directed_Distance_to_Target_C',
    'T_ext', 'T_ave_other_rooms',
    'Thermal_Source_T_C', 'Directed_Source_Room_DeltaT_C', 'Thermal_Input_Proxy',
    'thermal_mode_sign',
    'Room_dTdt_now_bwd2_C_per_min', 'Room_d2Tdt2_now_bwd2_C_per_min2',
    'hour_decimal','day_of_year','day_of_week','month','is_weekend',
    'doy_sin','doy_cos','dow_sin','dow_cos','month_sin','month_cos',
    'solar_declination_proxy_deg','daylight_hours_proxy','sunrise_hour_proxy','sunset_hour_proxy',
    'daylight_fraction_proxy','time_since_sunrise_h_proxy','time_to_sunset_h_proxy','solar_elevation_proxy',
]
COMMON_INPUTS += [f'Room_dTdt_central_-{lag}min_C_per_min' for lag in DERIVATIVE_HISTORY_LAGS_MINUTES]
COMMON_INPUTS += [f'Room_d2Tdt2_central_-{lag}min_C_per_min2' for lag in DERIVATIVE_HISTORY_LAGS_MINUTES]
COMMON_INPUTS += [f'ThermalSource_dTdt_central_-{lag}min_C_per_min' for lag in SOURCE_DERIVATIVE_HISTORY_LAGS_MINUTES]
COMMON_INPUTS += [f'ThermalSource_d2Tdt2_central_-{lag}min_C_per_min2' for lag in SOURCE_DERIVATIVE_HISTORY_LAGS_MINUTES]

# Explicitly forbidden future/label/identity variables.
FORBIDDEN_EXACT = {
    'room','room_label','event_id','event_key','shared_event_uid','target_Room_T','target_timestamp',
    'target_directed_rate_C_per_min','target_Room_dTdt_central_C_per_min',
    'remaining_time_to_end_min','remaining_duration_min','true_remaining_duration_min',
    'remaining_trajectory_quality_score','remaining_forward_progress_C','remaining_reverse_motion_C',
    'remaining_total_variation_C','remaining_quality_n_steps','rampup_start_time','rampup_end_time',
    'rampup_end_grid_time','extraction_end_time','split','split_block',
}
forbidden_pattern = re.compile(r'^Room_T -\d+min$|^Room_T \+\d+min$|^room__', re.I)

train_all=DATA[(DATA['split']=='train') & DATA['use_for_temperature_training']].copy()
anchor_train_all=ANCHORS[ANCHORS['split'].eq('train')].copy()
selected=[]; audit=[]
for c in COMMON_INPUTS:
    if c in FORBIDDEN_EXACT or forbidden_pattern.search(c):
        audit.append({'feature':c,'status':'forbidden'}); continue
    if c not in DATA.columns:
        audit.append({'feature':c,'status':'missing_LONG'}); continue
    if c!='horizon_min' and c not in ANCHORS.columns:
        audit.append({'feature':c,'status':'missing_duration_anchor'}); continue
    tr=pd.to_numeric(train_all[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
    if tr.notna().sum()==0:
        audit.append({'feature':c,'status':'all_missing_LONG_train'}); continue
    if c!='horizon_min':
        atr=pd.to_numeric(anchor_train_all[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
        if atr.notna().sum()==0:
            audit.append({'feature':c,'status':'all_missing_duration_anchor_train'}); continue
    phase_counts={p:int(pd.to_numeric(train_all.loc[train_all['phase'].eq(p),c],errors='coerce').replace([np.inf,-np.inf],np.nan).notna().sum()) for p in PHASES}
    if any(v==0 for v in phase_counts.values()):
        audit.append({'feature':c,'status':'missing_in_one_phase_train',**{f'finite_{p}':v for p,v in phase_counts.items()}}); continue
    if tr.dropna().nunique()<=1 and c!='thermal_mode_sign':
        audit.append({'feature':c,'status':'zero_variance_train'}); continue
    selected.append(c); audit.append({'feature':c,'status':'selected','missing_fraction_train':float(tr.isna().mean()),**{f'finite_{p}':v for p,v in phase_counts.items()}})

# thermal_mode_sign must vary in the combined training set; this confirms one network
# sees both modes.
if 'thermal_mode_sign' not in selected: selected.append('thermal_mode_sign')
if any(forbidden_pattern.search(c) for c in selected): raise RuntimeError('Raw room-temperature lag or room identity slipped into model inputs.')
if any(c in FORBIDDEN_EXACT for c in selected): raise RuntimeError('Forbidden target/identity input slipped into model inputs.')

pd.DataFrame(audit).to_csv(AUDIT_DIR/'unified_model_input_audit.csv',index=False)
pd.DataFrame({'feature':selected}).to_csv(AUDIT_DIR/'unified_model_inputs_ordered.csv',index=False)
MODEL_INPUTS=selected
print('Selected shared numeric inputs:',len(MODEL_INPUTS))
print(MODEL_INPUTS)


In [ ]:

# Fit training-only preprocessing and mode-specific target scales

# Mode-aware imputation is fitted from training trajectory rows only.
PHASE_MEDIANS={}
for phase in PHASES:
    z=train_all.loc[train_all['phase'].eq(phase),MODEL_INPUTS].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
    PHASE_MEDIANS[phase]={c:float(z[c].median()) for c in MODEL_INPUTS}
    bad=[c for c,v in PHASE_MEDIANS[phase].items() if not np.isfinite(v)]
    if bad: raise RuntimeError(f'{phase}: selected inputs still have no finite TRAIN median: {bad}')

def fill_phase_medians(df):
    x=df[MODEL_INPUTS].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan).copy()
    if 'phase' not in df.columns: raise RuntimeError('phase is required for phase-aware imputation.')
    phase_values=df['phase'].astype(str).to_numpy()
    for phase in PHASES:
        idx=np.where(phase_values==phase)[0]
        if len(idx)==0: continue
        for c in MODEL_INPUTS:
            col=x.columns.get_loc(c)
            vals=x.iloc[idx,col]
            x.iloc[idx,col]=vals.fillna(PHASE_MEDIANS[phase][c]).to_numpy()
    if x.isna().any().any(): raise RuntimeError('Phase-aware imputation left NaN values.')
    return x.to_numpy(dtype=float)

Xfilled=fill_phase_medians(train_all)
SCALER=StandardScaler().fit(Xfilled)
PREPROCESSOR={'numeric_cols':MODEL_INPUTS,'phase_medians':PHASE_MEDIANS,'scaler':SCALER,'include_room_identity':False,'imputation':'phase_specific_LONG_train_median'}
joblib.dump(PREPROCESSOR,MODEL_DIR/'unified_numeric_preprocessor.joblib')

def transform_X(df):
    return SCALER.transform(fill_phase_medians(df)).astype(np.float32)

def safe_std(x,floor):
    v=pd.to_numeric(pd.Series(x),errors='coerce').dropna().to_numpy(float)
    if len(v)<2: return float(floor)
    return float(max(float(np.std(v,ddof=0)),floor))

# Fit rate and temperature scales on training trajectory targets and duration scales on
# unique training prediction rows.
LOSS_SCALES={}
SOFT_DIRECTION_BETA_BY_PHASE={}
for phase in PHASES:
    tr=train_all[train_all['phase'].eq(phase)].copy()
    ar=ANCHORS[(ANCHORS['phase']==phase)&(ANCHORS['split']=='train')].copy()
    rate_scale=safe_std(tr['target_directed_rate_C_per_min'],1e-4)
    temp_scale=safe_std(tr['target_Room_T'],0.05)
    duration_scale=safe_std(ar['true_remaining_duration_min'],1.0)
    med_step=float(np.nanmedian(np.abs(pd.to_numeric(tr['target_directed_rate_C_per_min'],errors='coerce').to_numpy(float)*STEP_MIN)))
    if not np.isfinite(med_step): med_step=0.05
    beta_dir=max(SOFT_DIRECTION_BETA_FLOOR_C,0.10*med_step)
    LOSS_SCALES[phase]={'rate_C_per_min':rate_scale,'temperature_C':temp_scale,'remaining_min':duration_scale}
    SOFT_DIRECTION_BETA_BY_PHASE[phase]=float(beta_dir)

with open(MODEL_DIR/'loss_scales_train_only_by_phase.json','w') as f: json.dump(LOSS_SCALES,f,indent=2)
with open(MODEL_DIR/'soft_crossing_parameters.json','w') as f: json.dump({'temp_beta_C':SOFT_TEMP_BETA_C,'direction_beta_C_by_phase':SOFT_DIRECTION_BETA_BY_PHASE},f,indent=2)
print('TRAIN-only loss scales:'); display(pd.DataFrame(LOSS_SCALES).T)
print('Soft direction beta:',SOFT_DIRECTION_BETA_BY_PHASE)

# Score distribution is shown before any model is trained.
score_summary=(ANCHORS.groupby(['phase','split'])['remaining_trajectory_quality_score']
               .agg(['count','mean','median','std','min','max']).reset_index())
score_summary.to_csv(TABLE_DIR/'duration_quality_score_distribution.csv',index=False)
display(score_summary.round(4))


In [ ]:
# A room/mode pair can have validation or test rows but no eligible training rows.
# Retain the chronological split and leave unsupported local-model predictions missing.
# Fit each local tree model on its eligible training rows.
# Compare methods only where they all return valid predictions, and report neural
# coverage separately.

ET_ROOT = MODEL_DIR/'room_phase_extra_trees_temperature'
ET_ROOT.mkdir(exist_ok=True)
ET_PRED = np.full(len(DATA), np.nan, dtype=float)
manifest = []
support_rows = []

for phase in PHASES:
    rooms = sorted(
        DATA.loc[DATA['phase'].eq(phase), 'room']
        .dropna().astype(str).unique()
    )

    for room in rooms:
        mask_all = (
            DATA['phase'].eq(phase)
            & DATA['room'].eq(room)
            & DATA['use_for_temperature_training']
        )
        tr = DATA[mask_all & DATA['split'].eq('train')].copy()
        va = DATA[mask_all & DATA['split'].eq('valid')].copy()
        te = DATA[mask_all & DATA['split'].eq('test')].copy()

        y_all = (
            pd.to_numeric(tr['target_Room_T'], errors='coerce')
            - pd.to_numeric(tr['Room_T'], errors='coerce')
        ).to_numpy(float)
        good_y = np.isfinite(y_all)
        n_train_usable = int(good_y.sum())

        supported = n_train_usable >= MIN_ET_TRAIN_ROWS_PER_ROOM
        if supported:
            reason = 'supported'
        elif n_train_usable == 0:
            reason = 'no_usable_train_rows_under_authoritative_split'
        else:
            reason = f'usable_train_rows_below_minimum_{MIN_ET_TRAIN_ROWS_PER_ROOM}'

        support_rows.append({
            'phase': phase,
            'room': room,
            'supported': bool(supported),
            'reason': reason,
            'train_rows_raw': int(len(tr)),
            'train_rows_usable_target': n_train_usable,
            'valid_rows': int(len(va)),
            'test_rows': int(len(te)),
            'train_events': int(tr['event_key'].nunique()) if len(tr) else 0,
            'valid_events': int(va['event_key'].nunique()) if len(va) else 0,
            'test_events': int(te['event_key'].nunique()) if len(te) else 0,
        })

        if not supported:
            print(
                f'SKIP room-specific temperature ET: {phase}/{room} | '
                f'usable TRAIN rows={n_train_usable}, valid rows={len(va)}, '
                f'test rows={len(te)} | {reason}'
            )
            continue

        # Use only rows with a finite supervised target for this baseline fit.
        tr_fit = tr.loc[good_y].copy()
        y_fit = y_all[good_y]
        X_fit_num = tr_fit[MODEL_INPUTS].apply(pd.to_numeric, errors='coerce').replace([np.inf,-np.inf],np.nan)

        # Use room-specific training medians, falling back to operating-mode training
        # medians for entirely missing features.
        for c in MODEL_INPUTS:
            room_med = pd.to_numeric(X_fit_num[c], errors='coerce').median()
            fill_value = float(room_med) if np.isfinite(room_med) else float(PHASE_MEDIANS[phase][c])
            X_fit_num[c] = X_fit_num[c].fillna(fill_value)

        imp = SimpleImputer(strategy='median', keep_empty_features=True).fit(X_fit_num)
        X_fit = imp.transform(X_fit_num)

        model = ExtraTreesRegressor(**ET_PARAMS).fit(X_fit, y_fit)

        # Predict this pair on every split only after fitting strictly on TRAIN.
        target_idx = DATA.index[mask_all].to_numpy()
        X_all = imp.transform(
            DATA.loc[target_idx, MODEL_INPUTS].apply(pd.to_numeric, errors='coerce')
        )
        base_T = pd.to_numeric(
            DATA.loc[target_idx, 'Room_T'], errors='coerce'
        ).to_numpy(float)
        ET_PRED[target_idx] = base_T + model.predict(X_all)

        phase_dir = ET_ROOT/phase
        phase_dir.mkdir(exist_ok=True)
        artifact = phase_dir/f'{room}.joblib'
        joblib.dump({
            'model': model,
            'imputer': imp,
            'feature_names': MODEL_INPUTS,
            'phase': phase,
            'room': room,
            'train_rows_usable': n_train_usable,
        }, artifact)

        manifest.append({
            'phase': phase,
            'room': room,
            'train_rows_raw': int(len(tr)),
            'train_rows_usable': n_train_usable,
            'artifact_path': str(artifact),
        })

DATA['pred_T_RoomPhase_ExtraTrees'] = ET_PRED

ET_SUPPORTED_PAIRS = {
    (str(r['phase']), str(r['room'])) for r in manifest
}
DATA['ET_room_phase_supported'] = [
    (str(p), str(r)) in ET_SUPPORTED_PAIRS
    for p, r in zip(DATA['phase'], DATA['room'])
]

support_df = pd.DataFrame(support_rows)
support_df.to_csv(ET_ROOT/'room_phase_support_audit.csv', index=False)
support_df.to_csv(AUDIT_DIR/'room_phase_temperature_ET_support_audit.csv', index=False)

manifest_df = pd.DataFrame(
    manifest,
    columns=['phase','room','train_rows_raw','train_rows_usable','artifact_path']
)
manifest_df.to_csv(ET_ROOT/'model_manifest.csv', index=False)

if len(manifest_df) == 0:
    raise RuntimeError(
        'No room/phase Extra Trees model has sufficient TRAIN support. '
        'The unified NN is still conceptually valid, but the requested ET baseline '
        'cannot be formed from this split.'
    )

model_rows = DATA['use_for_temperature_training']
n_total = int(model_rows.sum())
n_et = int(DATA.loc[model_rows, 'pred_T_RoomPhase_ExtraTrees'].notna().sum())

print('\nExtra Trees room/phase models trained:', len(manifest_df))
print('Usable temperature rows:', n_total)
print('Rows covered by room-specific ET:', n_et)
print('Rows intentionally unsupported by room-specific ET:', n_total - n_et)
print('\nRoom/phase support audit:')
display(support_df.sort_values(['phase','supported','room']).reset_index(drop=True))


In [ ]:

# Build two supervision streams for the shared NN
#    A) long-format trajectories -> directed-rate + reconstructed-T loss
#    B) saved duration anchors -> remaining-time loss

DATA_X=transform_X(DATA)

class TemperatureTrajectoryDataset(Dataset):
    def __init__(self,frame,Xall):
        self.items=[]
        d=frame[frame['use_for_temperature_training']].copy()
        for (phase,event_key,timestamp),g in d.groupby(['phase','event_key','timestamp'],sort=False):
            g=g.sort_values('horizon_min').drop_duplicates('horizon_min',keep='first')
            h=pd.to_numeric(g['horizon_min'],errors='coerce').to_numpy(float)
            valid=np.isfinite(h)&(h>0)
            g=g.iloc[np.where(valid)[0]].copy(); h=h[valid]
            if len(g)==0: continue
            order=np.argsort(h); g=g.iloc[order].copy(); h=h[order]
            if np.any(np.abs(h/STEP_MIN-np.round(h/STEP_MIN))>1e-8): continue
            rid=g['_row_id'].to_numpy(int)
            target_rate=pd.to_numeric(g['target_directed_rate_C_per_min'],errors='coerce').to_numpy(float)
            target_T=pd.to_numeric(g['target_Room_T'],errors='coerce').to_numpy(float)
            rate_mask=np.isfinite(target_rate); temp_mask=np.isfinite(target_T)
            if not temp_mask.any() or not rate_mask.any(): continue
            first=g.iloc[0]
            v0_phys=pd.to_numeric(first.get('Room_dTdt_now_bwd2_C_per_min',np.nan),errors='coerce')
            if not np.isfinite(v0_phys):
                past=pd.to_numeric(first.get('Room_dTdt_central_-5min_C_per_min',np.nan),errors='coerce')
                v0_phys=float(past) if np.isfinite(past) else 0.0
            self.items.append({
                'phase':str(phase),'event_key':str(event_key),'timestamp':pd.Timestamp(timestamp),
                'row_ids':rid,'X':Xall[rid],'h':h.astype(np.float32),
                'target_rate':target_rate.astype(np.float32),'rate_mask':rate_mask,
                'target_T':target_T.astype(np.float32),'temp_mask':temp_mask,
                'T0':float(first['Room_T']),'sign':float(first['thermal_mode_sign']),'v0_phys':float(v0_phys),
            })
    def __len__(self): return len(self.items)
    def __getitem__(self,i): return self.items[i]

def horizon_grid(phase):
    return np.arange(STEP_MIN,DURATION_MAX_HORIZON_BY_PHASE[phase]+1e-9,STEP_MIN,dtype=float)

def repeat_anchor_for_horizons(row,phase):
    h=horizon_grid(phase)
    syn=pd.DataFrame([row.to_dict()]*len(h))
    syn['horizon_min']=h
    syn['target_timestamp']=pd.Timestamp(row['feature_source_timestamp'])+pd.to_timedelta(h,unit='m')
    return h,syn

class DurationAnchorDataset(Dataset):
    def __init__(self,frame):
        self.items=[]
        for _,row in frame.iterrows():
            phase=str(row['phase'])
            true_rem=pd.to_numeric(row['true_remaining_duration_min'],errors='coerce')
            T0=pd.to_numeric(row['Room_T'],errors='coerce')
            sign=pd.to_numeric(row['thermal_mode_sign'],errors='coerce')
            target=pd.to_numeric(row['Phase_Target_T_C'],errors='coerce')
            gap=pd.to_numeric(row['asof_gap_min'],errors='coerce')
            q=pd.to_numeric(row['remaining_trajectory_quality_score'],errors='coerce')
            if not all(np.isfinite(v) for v in [true_rem,T0,sign,target,gap,q]): continue
            h,syn=repeat_anchor_for_horizons(row,phase)
            try:
                X=transform_X(syn)
            except Exception as e:
                raise RuntimeError(f"Failed to build synthetic inputs for anchor {row.get('anchor_key','?')}: {e}") from e
            v0=pd.to_numeric(row.get('Room_dTdt_now_bwd2_C_per_min',np.nan),errors='coerce')
            if not np.isfinite(v0):
                v0=pd.to_numeric(row.get('Room_dTdt_central_-5min_C_per_min',0.0),errors='coerce')
                if not np.isfinite(v0): v0=0.0
            self.items.append({
                'phase':phase,'anchor_key':str(row['anchor_key']),'event_key':str(row['event_key']),
                'anchor_kind':str(row.get('anchor_kind','anchor')),'X':X,'h':h.astype(np.float32),
                'T0':float(T0),'sign':float(sign),'v0_phys':float(v0),'phase_target':float(target),
                'true_remaining':float(true_rem),'asof_gap':float(gap),'q':float(np.clip(q,0,1)),
            })
    def __len__(self): return len(self.items)
    def __getitem__(self,i): return self.items[i]

TEMP_DATASETS={}
DURATION_DATASETS={}
for split in ['train','valid','test']:
    for phase in PHASES:
        tds=TemperatureTrajectoryDataset(DATA[(DATA['split']==split)&(DATA['phase']==phase)],DATA_X)
        TEMP_DATASETS[(split,phase)]=tds
        ads=DurationAnchorDataset(ANCHORS[(ANCHORS['split']==split)&(ANCHORS['phase']==phase)])
        DURATION_DATASETS[(split,phase)]=ads
        print(split,phase,'temperature issue trajectories:',len(tds),'| authoritative duration anchors:',len(ads))
        if split in ['train','valid'] and (len(tds)==0 or len(ads)==0):
            raise RuntimeError(f'Empty required supervision dataset for {split}/{phase}')

# Verify that duration supervision covers every required prediction row.
anchor_audit=[]
for split in ['train','valid','test']:
    for phase in PHASES:
        expected=int(ANCHORS[(ANCHORS['split']==split)&(ANCHORS['phase']==phase)]['anchor_key'].nunique())
        actual=len(DURATION_DATASETS[(split,phase)])
        anchor_audit.append({'phase':phase,'split':split,'authoritative_anchor_count':expected,'duration_dataset_count':actual,'difference':actual-expected})
anchor_audit=pd.DataFrame(anchor_audit)
anchor_audit.to_csv(AUDIT_DIR/'authoritative_duration_supervision_coverage.csv',index=False)
display(anchor_audit)
if (anchor_audit['difference']!=0).any():
    raise RuntimeError('Some authoritative duration anchors were lost while building duration supervision; inspect audit before training.')


In [ ]:

# Unified directed-rate NN + fully vectorized GPU losses

class UnifiedDirectedRateNN(nn.Module):
    def __init__(self,n_in,hidden=(128,64,32),dropout=0.10):
        super().__init__(); layers=[]; prev=n_in
        for h in hidden:
            layers += [nn.Linear(prev,h),nn.Tanh()]
            if dropout>0: layers.append(nn.Dropout(dropout))
            prev=h
        layers.append(nn.Linear(prev,1)); self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x).squeeze(-1)

def _model_forward_fp32(model,x):
    # Use mixed precision for network operations and 32-bit floats for integration and
    # losses.
    with torch.autocast(device_type=DEVICE.type,dtype=AMP_DTYPE,enabled=USE_AMP):
        y=model(x)
    return y.float()

def make_batches(ds,batch_size,shuffle=True,seed=0):
    idx=np.arange(len(ds))
    if shuffle: np.random.default_rng(seed).shuffle(idx)
    return [[ds.items[j] for j in idx[i:i+batch_size]] for i in range(0,len(idx),batch_size)]

def stack_trajectory_batch(batch):
    B=len(batch); H=max(len(x['h']) for x in batch); F=batch[0]['X'].shape[1]
    X=np.zeros((B,H,F),np.float32); h=np.zeros((B,H),np.float32)
    target_rate=np.zeros((B,H),np.float32); target_T=np.zeros((B,H),np.float32)
    valid=np.zeros((B,H),bool); rmask=np.zeros((B,H),bool); tmask=np.zeros((B,H),bool)
    row_ids=[]
    T0=np.zeros(B,np.float32); sign=np.zeros(B,np.float32); v0=np.zeros(B,np.float32)
    for b,item in enumerate(batch):
        n=len(item['h']); X[b,:n]=item['X']; h[b,:n]=item['h']; valid[b,:n]=True
        if n<H: h[b,n:]=item['h'][-1]  # Repeated padded endpoints contribute zero integration time.
        target_rate[b,:n]=item['target_rate']; target_T[b,:n]=item['target_T']
        rmask[b,:n]=item['rate_mask']; tmask[b,:n]=item['temp_mask']
        T0[b]=item['T0']; sign[b]=item['sign']; v0[b]=item['v0_phys']; row_ids.append(item['row_ids'])
    def tt(a,dtype=torch.float32): return torch.as_tensor(a,dtype=dtype,device=DEVICE)
    return {'X':tt(X),'h':tt(h),'target_rate':tt(target_rate),'target_T':tt(target_T),
            'valid':tt(valid,torch.bool),'rmask':tt(rmask,torch.bool),'tmask':tt(tmask,torch.bool),
            'T0':tt(T0),'sign':tt(sign),'v0':tt(v0),'row_ids':row_ids}

def integrate_variable_batch_torch(h,pred_directed,T0,sign,v0):
    pred_phys=sign[:,None]*pred_directed.float()
    dt=torch.cat([h[:,:1],h[:,1:]-h[:,:-1]],dim=1).float()
    prev_rate=torch.cat([v0[:,None],pred_phys[:,:-1]],dim=1)
    increments=0.5*dt*(prev_rate+pred_phys)
    return T0[:,None]+torch.cumsum(increments,dim=1)

def trajectory_batch_forward(model,batch):
    z=stack_trajectory_batch(batch); B,H,F=z['X'].shape
    pred=_model_forward_fp32(model,z['X'].reshape(B*H,F)).reshape(B,H)
    pred_T=integrate_variable_batch_torch(z['h'],pred,z['T0'],z['sign'],z['v0'])
    return z,pred,pred_T

def trajectory_batch_loss(model,batch,phase):
    z,pred,pred_T=trajectory_batch_forward(model,batch)
    rs=float(LOSS_SCALES[phase]['rate_C_per_min']); ts=float(LOSS_SCALES[phase]['temperature_C'])
    rm=z['rmask'] & z['valid']; tm=z['tmask'] & z['valid']
    if not rm.any() or not tm.any(): raise RuntimeError(f'{phase}: empty trajectory supervision mask.')
    Lr=torch.mean(((pred[rm]-z['target_rate'][rm])/rs)**2)
    Lt=torch.mean(((pred_T[tm]-z['target_T'][tm])/ts)**2)
    return {'trajectory':0.5*(Lr+Lt),'rate':Lr,'temperature':Lt}

def stack_duration_batch(batch):
    X=torch.as_tensor(np.stack([x['X'] for x in batch]).astype(np.float32),device=DEVICE)
    h=torch.as_tensor(batch[0]['h'],dtype=torch.float32,device=DEVICE)
    if any(not np.array_equal(x['h'],batch[0]['h']) for x in batch): raise RuntimeError('Duration batch horizon grids differ.')
    def tv(k): return torch.as_tensor([x[k] for x in batch],dtype=torch.float32,device=DEVICE)
    return {'X':X,'h':h,'T0':tv('T0'),'sign':tv('sign'),'v0':tv('v0_phys'),'target':tv('phase_target'),
            'true':tv('true_remaining'),'gap':tv('asof_gap'),'q':tv('q')}

def integrate_fixed_batch_torch(h,pred_directed,T0,sign,v0):
    pred_phys=sign[:,None]*pred_directed.float()
    dt=torch.cat([h[:1],h[1:]-h[:-1]]).float()
    prev_rate=torch.cat([v0[:,None],pred_phys[:,:-1]],dim=1)
    return T0[:,None]+torch.cumsum(0.5*dt[None,:]*(prev_rate+pred_phys),dim=1)

def soft_crossing_batch_torch(h,pred_T,T0,target,sign,v0,phase):
    threshold=target-sign*END_BAND_C
    temp_beta=float(SOFT_TEMP_BETA_C); dir_beta=float(SOFT_DIRECTION_BETA_BY_PHASE[phase])
    a0=torch.sigmoid(sign*(T0-threshold)/temp_beta)*torch.sigmoid(sign*v0*STEP_MIN/dir_beta)
    prev_T=torch.cat([T0[:,None],pred_T[:,:-1]],dim=1)
    a=torch.sigmoid(sign[:,None]*(pred_T-threshold[:,None])/temp_beta)*torch.sigmoid(sign[:,None]*(pred_T-prev_T)/dir_beta)
    one_minus=torch.clamp(1.0-a,min=1e-7,max=1.0)
    prefix=torch.cumprod(one_minus,dim=1)
    survival_before=torch.cat([(1-a0)[:,None],(1-a0)[:,None]*prefix[:,:-1]],dim=1)
    p_first=survival_before*a
    survival_after=(1-a0)*prefix[:,-1]
    return torch.sum(p_first*h[None,:],dim=1)+survival_after*h[-1]

def duration_batch_forward(model,batch,phase):
    z=stack_duration_batch(batch); B,H,F=z['X'].shape
    pred=_model_forward_fp32(model,z['X'].reshape(B*H,F)).reshape(B,H)
    pred_T=integrate_fixed_batch_torch(z['h'],pred,z['T0'],z['sign'],z['v0'])
    pred_source=soft_crossing_batch_torch(z['h'],pred_T,z['T0'],z['target'],z['sign'],z['v0'],phase)
    pred_anchor=torch.clamp(pred_source-z['gap'],min=0.0)
    return z,pred_anchor,pred_T,pred

def duration_batch_loss(model,batch,phase):
    z,pred,_,_=duration_batch_forward(model,batch,phase)
    ds=float(LOSS_SCALES[phase]['remaining_min'])
    sq=((pred-z['true'])/ds)**2
    w=z['q']  # Weight the training duration loss by trajectory regularity; assess its contribution in Notebook 05.
    return torch.sum(w*sq)/(torch.sum(w)+1e-8)

def evaluate_trajectory(model,ds,phase,batch_size=512):
    model.eval(); sums={'trajectory':0.0,'rate':0.0,'temperature':0.0}; weight=0
    with torch.inference_mode():
        for b in make_batches(ds,batch_size,False):
            z=trajectory_batch_loss(model,b,phase); n=len(b); weight+=n
            for k in sums: sums[k]+=float(z[k].detach().cpu())*n
    return {k:v/max(weight,1) for k,v in sums.items()}

def predict_duration_soft_dataset(model,ds,phase,batch_size=1024):
    model.eval(); rows=[]
    with torch.inference_mode():
        for batch in make_batches(ds,batch_size,False):
            z,pred,_,_=duration_batch_forward(model,batch,phase); pv=pred.cpu().numpy()
            for item,p in zip(batch,pv):
                rows.append({'anchor_key':item['anchor_key'],'event_key':item['event_key'],'anchor_kind':item['anchor_kind'],
                             'phase':phase,'true_remaining_duration_min':item['true_remaining'],
                             'pred_remaining_duration_min':float(p),'q':item['q']})
    return pd.DataFrame(rows)

def duration_eval_metrics(model,ds,phase):
    d=predict_duration_soft_dataset(model,ds,phase)
    e=d['pred_remaining_duration_min'].to_numpy(float)-d['true_remaining_duration_min'].to_numpy(float)
    denom=np.maximum(d['true_remaining_duration_min'].to_numpy(float),STEP_MIN)
    dscl=float(LOSS_SCALES[phase]['remaining_min'])
    return {
        'duration_norm_mse_unweighted':float(np.mean((e/dscl)**2)),
        'duration_MAE_min_unweighted':float(np.mean(np.abs(e))),
        'duration_RMSE_min_unweighted':float(np.sqrt(np.mean(e**2))),
        'duration_Bias_min_unweighted':float(np.mean(e)),
        'duration_Median_RAE_pct_unweighted':float(100*np.median(np.abs(e)/denom)),
    },d

# Fast implementation forward/backward check: finite forward/backward on both modes.
_test_model=UnifiedDirectedRateNN(len(MODEL_INPUTS),NN_HIDDEN_DIMS,0.0).to(DEVICE)
for _phase in PHASES:
    _b=DURATION_DATASETS[('train',_phase)].items[:min(8,len(DURATION_DATASETS[('train',_phase)].items))]
    _loss=duration_batch_loss(_test_model,_b,_phase)
    if not torch.isfinite(_loss): raise RuntimeError(f'{_phase}: non-finite vectorized duration loss.')
print('GPU-vectorized trajectory/duration utilities ready.')


In [ ]:

# Regularity-weighted training — scheduler, resume, best validation loss

RESUME_PATH=CHECKPOINT_DIR/'resume_state.pt'
BEST_PATH=MODEL_DIR/'unified_directed_rate_nn.pt'
HISTORY_PATH=TABLE_DIR/'unified_nn_training_history_score_weighted.csv'
COMPLETE_PATH=CHECKPOINT_DIR/'TRAINING_COMPLETE.json'

def capture_rng_state():
    st={'numpy':np.random.get_state(),'python':random.getstate(),'torch':torch.get_rng_state()}
    if torch.cuda.is_available(): st['cuda']=torch.cuda.get_rng_state_all()
    return st

def restore_rng_state(st):
    if not st: return
    np.random.set_state(st['numpy']); random.setstate(st['python']); torch.set_rng_state(st['torch'])
    if torch.cuda.is_available() and 'cuda' in st: torch.cuda.set_rng_state_all(st['cuda'])

def save_best_model(model,epoch,val_monitor,extra):
    torch.save({
        'state_dict':{k:v.detach().cpu() for k,v in model.state_dict().items()},
        'n_inputs':len(MODEL_INPUTS),'hidden_dims':NN_HIDDEN_DIMS,'dropout':NN_DROPOUT,
        'score_mode':'score_weighted','best_epoch':int(epoch),'best_validation_duration_norm_mse':float(val_monitor),
        'shared_split_sha256':SHARED_SPLIT_SHA256,'extra':extra,
    },BEST_PATH)

def train_final_model():
    reset_seeds(RANDOM_SEED)
    model=UnifiedDirectedRateNN(len(MODEL_INPUTS),NN_HIDDEN_DIMS,NN_DROPOUT).to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=NN_LR,weight_decay=NN_WEIGHT_DECAY)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,mode='min',factor=NN_SCHEDULER_FACTOR,
        patience=NN_SCHEDULER_PATIENCE_VALIDATIONS,min_lr=NN_MIN_LR
    )
    scaler=torch.cuda.amp.GradScaler(enabled=USE_AMP)

    start_epoch=1; best_val=np.inf; best_epoch=0; stale_validations=0; history=[]
    if RESUME_TRAINING and RESUME_PATH.exists():
        ck=safe_torch_load(RESUME_PATH,map_location=DEVICE)
        if ck.get('shared_split_sha256')!=SHARED_SPLIT_SHA256:
            raise RuntimeError('Resume checkpoint was trained with a different chronological split.')
        model.load_state_dict(ck['model_state_dict']); optimizer.load_state_dict(ck['optimizer_state_dict']); scheduler.load_state_dict(ck['scheduler_state_dict'])
        if ck.get('scaler_state_dict') and USE_AMP: scaler.load_state_dict(ck['scaler_state_dict'])
        start_epoch=int(ck['epoch'])+1; best_val=float(ck['best_val']); best_epoch=int(ck['best_epoch']); stale_validations=int(ck['stale_validations']); history=ck.get('history',[])
        restore_rng_state(ck.get('rng_state'))
        print(f'Resuming from epoch {start_epoch} | best validation loss={best_val:.6f} at epoch {best_epoch}')

    if COMPLETE_PATH.exists() and BEST_PATH.exists() and not FORCE_CONTINUE_AFTER_EARLY_STOP:
        print('Training already marked complete; loading the existing best-validation model.')
        ck=safe_torch_load(BEST_PATH,map_location=DEVICE); model.load_state_dict(ck['state_dict']); return model,pd.DataFrame(history)

    for epoch in range(start_epoch,NN_MAX_EPOCHS+1):
        t_epoch=time.time(); model.train()
        tb={p:make_batches(TEMP_DATASETS[('train',p)],NN_BATCH_TRAJECTORIES_PER_PHASE,True,RANDOM_SEED+epoch+(0 if p=='heating' else 10000)) for p in PHASES}
        db={p:make_batches(DURATION_DATASETS[('train',p)],NN_BATCH_DURATION_ANCHORS_PER_PHASE,True,RANDOM_SEED+20000+epoch+(0 if p=='heating' else 10000)) for p in PHASES}
        nsteps=max(*[len(tb[p]) for p in PHASES],*[len(db[p]) for p in PHASES]); cyc_t={p:cycle(tb[p]) for p in PHASES}; cyc_d={p:cycle(db[p]) for p in PHASES}
        acc=[]
        for _ in range(nsteps):
            optimizer.zero_grad(set_to_none=True)
            phase_losses={}
            for phase in PHASES:
                lt=trajectory_batch_loss(model,next(cyc_t[phase]),phase)
                ld=duration_batch_loss(model,next(cyc_d[phase]),phase)
                phase_losses[phase]={'trajectory':lt['trajectory'],'rate':lt['rate'],'temperature':lt['temperature'],'duration':ld,'total':lt['trajectory']+ld}
            total=0.5*(phase_losses['heating']['total']+phase_losses['cooling']['total'])
            scaler.scale(total).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); scaler.step(optimizer); scaler.update()
            acc.append({
                'total':float(total.detach().cpu()),
                'trajectory':float((0.5*(phase_losses['heating']['trajectory']+phase_losses['cooling']['trajectory'])).detach().cpu()),
                'duration_q_weighted':float((0.5*(phase_losses['heating']['duration']+phase_losses['cooling']['duration'])).detach().cpu()),
            })
        tr={k:float(np.mean([x[k] for x in acc])) for k in acc[0]}
        rec={'epoch':epoch,**{f'train_{k}':v for k,v in tr.items()},'lr':float(optimizer.param_groups[0]['lr']),'epoch_seconds':float(time.time()-t_epoch),'validation_performed':False}

        do_val=(epoch==1 or epoch%NN_VALIDATION_EVERY==0 or epoch==NN_MAX_EPOCHS)
        if do_val:
            tval=time.time(); vt={p:evaluate_trajectory(model,TEMP_DATASETS[('valid',p)],p) for p in PHASES}; vd={}; pred={}
            for p in PHASES: vd[p],pred[p]=duration_eval_metrics(model,DURATION_DATASETS[('valid',p)],p)
            val_duration_loss=0.5*(vd['heating']['duration_norm_mse_unweighted']+vd['cooling']['duration_norm_mse_unweighted'])
            val_duration_mae=0.5*(vd['heating']['duration_MAE_min_unweighted']+vd['cooling']['duration_MAE_min_unweighted'])
            val_rel=0.5*(vd['heating']['duration_Median_RAE_pct_unweighted']+vd['cooling']['duration_Median_RAE_pct_unweighted'])
            val_traj=0.5*(vt['heating']['trajectory']+vt['cooling']['trajectory'])
            scheduler.step(val_duration_loss)
            rec.update({
                'validation_performed':True,'validation_seconds':float(time.time()-tval),
                'val_duration_norm_mse_unweighted_balanced':val_duration_loss,
                'val_duration_MAE_min_unweighted_balanced':val_duration_mae,
                'val_duration_Median_RAE_pct_unweighted_balanced':val_rel,
                'val_trajectory_balanced':val_traj,
                'val_heating_MAE_min':vd['heating']['duration_MAE_min_unweighted'],
                'val_cooling_MAE_min':vd['cooling']['duration_MAE_min_unweighted'],
                'val_heating_Median_RAE_pct':vd['heating']['duration_Median_RAE_pct_unweighted'],
                'val_cooling_Median_RAE_pct':vd['cooling']['duration_Median_RAE_pct_unweighted'],
                'lr_after_scheduler':float(optimizer.param_groups[0]['lr']),
            })
            improved=val_duration_loss < best_val-NN_MIN_DELTA
            if improved:
                best_val=val_duration_loss; best_epoch=epoch; stale_validations=0
                save_best_model(model,epoch,val_duration_loss,rec)
            else:
                stale_validations+=1
            print(f"epoch={epoch:03d} valDurLoss={val_duration_loss:.5f} traj={val_traj:.4f} MAE H/C={vd['heating']['duration_MAE_min_unweighted']:.2f}/{vd['cooling']['duration_MAE_min_unweighted']:.2f} min | MedRAE H/C={vd['heating']['duration_Median_RAE_pct_unweighted']:.1f}/{vd['cooling']['duration_Median_RAE_pct_unweighted']:.1f}% | lr={optimizer.param_groups[0]['lr']:.2e} | best={best_epoch}")
        history.append(rec)
        pd.DataFrame(history).to_csv(HISTORY_PATH,index=False)

        # Full resumable state is saved every epoch.
        torch.save({
            'epoch':epoch,'model_state_dict':model.state_dict(),'optimizer_state_dict':optimizer.state_dict(),'scheduler_state_dict':scheduler.state_dict(),
            'scaler_state_dict':scaler.state_dict() if USE_AMP else None,'best_val':best_val,'best_epoch':best_epoch,'stale_validations':stale_validations,
            'history':history,'rng_state':capture_rng_state(),'shared_split_sha256':SHARED_SPLIT_SHA256,
        },RESUME_PATH)

        if do_val and stale_validations>=NN_EARLY_STOP_PATIENCE_VALIDATIONS:
            print(f'Early stopping: validation duration loss did not improve for {stale_validations} validation checkpoints.')
            break

    if not BEST_PATH.exists(): raise RuntimeError('Training finished without a best-validation checkpoint.')
    with open(COMPLETE_PATH,'w') as f: json.dump({'best_epoch':best_epoch,'best_validation_duration_norm_mse':best_val},f,indent=2)
    best=safe_torch_load(BEST_PATH,map_location=DEVICE); model.load_state_dict(best['state_dict']); model.eval()
    return model,pd.DataFrame(history)

model,HISTORY=train_final_model()
SELECTED_SCORE_MODE='score_weighted'

# Compatibility metadata consumed by Notebook 03.
with open(MODEL_DIR/'score_weighting_model_selection.json','w') as f:
    json.dump({'selected_mode':'score_weighted','selection_metric':'balanced unweighted normalized validation duration MSE','test_used_for_selection':False,'best_checkpoint':str(BEST_PATH)},f,indent=2)

if len(HISTORY):
    hv=HISTORY[HISTORY.get('validation_performed',False)==True].copy()
    if len(hv):
        fig,axes=plt.subplots(1,2,figsize=(14,5),constrained_layout=True)
        axes[0].plot(hv['epoch'],hv['val_duration_norm_mse_unweighted_balanced'],marker='o',label='Validation duration loss')
        axes[0].plot(hv['epoch'],hv['val_trajectory_balanced'],marker='o',label='Validation trajectory loss')
        axes[0].set(xlabel='Epoch',ylabel='Dimensionless loss',title='Validation losses'); axes[0].grid(alpha=.25); axes[0].legend(frameon=False)
        axes[1].plot(hv['epoch'],hv['lr_after_scheduler'],marker='o'); axes[1].set_yscale('log'); axes[1].set(xlabel='Epoch',ylabel='Learning rate',title='ReduceLROnPlateau schedule'); axes[1].grid(alpha=.25)
        fig.savefig(FIG_DIR/'FINAL_training_validation_losses_and_learning_rate.png',dpi=300,bbox_inches='tight'); plt.show()


In [ ]:

# Batched reconstruction of selected final-NN temperature trajectories

def predict_nn_dataset(ds,model,batch_size=512):
    rows=[]; model.eval()
    with torch.inference_mode():
        for batch in make_batches(ds,batch_size,False):
            z,pv,pT=trajectory_batch_forward(model,batch)
            pv=pv.cpu().numpy(); pT=pT.cpu().numpy(); valid=z['valid'].cpu().numpy()
            for b,item in enumerate(batch):
                n=int(valid[b].sum())
                for rid,v,t in zip(item['row_ids'][:n],pv[b,:n],pT[b,:n]):
                    rows.append({'_row_id':int(rid),'pred_directed_rate_UnifiedNN':float(v),'pred_T_UnifiedNN':float(t)})
    return pd.DataFrame(rows)

parts=[]
for split_name in ['train','valid','test']:
    for phase in PHASES: parts.append(predict_nn_dataset(TEMP_DATASETS[(split_name,phase)],model))
NNP=pd.concat(parts,ignore_index=True).drop_duplicates('_row_id')
DATA=DATA.merge(NNP,on='_row_id',how='left',validate='one_to_one')

missing_nn=DATA.loc[DATA['use_for_temperature_training'],'pred_T_UnifiedNN'].isna()
if missing_nn.any():
    bad=DATA.loc[DATA['use_for_temperature_training']&DATA['pred_T_UnifiedNN'].isna(),['phase','room','event_key','split','timestamp','horizon_min']]
    bad.to_csv(AUDIT_DIR/'BAD_unified_NN_missing_temperature_predictions.csv',index=False)
    raise RuntimeError(f'Unified NN failed to cover {int(missing_nn.sum())} usable temperature rows.')

DATA.to_csv(PRED_DIR/'temperature_predictions_wide.csv',index=False)
base_cols=['_row_id','phase','room','event_key','split','target_context','timestamp','target_timestamp','horizon_min','Room_T','target_Room_T','target_directed_rate_C_per_min','rampup_end_time','Phase_Target_T_C','thermal_mode_sign','ET_room_phase_supported']
base_cols=[c for c in base_cols if c in DATA.columns]
long=[]
for method,col in [('Room-specific Extra Trees','pred_T_RoomPhase_ExtraTrees'),('Unified derivative NN','pred_T_UnifiedNN')]:
    x=DATA[base_cols].copy(); x['method']=method; x['pred_T']=pd.to_numeric(DATA[col],errors='coerce')
    x['error_C']=x['pred_T']-pd.to_numeric(x['target_Room_T'],errors='coerce'); x['absolute_error_C']=x['error_C'].abs(); long.append(x)
PRED_LONG=pd.concat(long,ignore_index=True); PRED_LONG.to_csv(PRED_DIR/'all_temperature_predictions_long.csv',index=False)


In [ ]:

# Temperature diagnostics — exact common split/support + full 0–60 cooling axis

def temperature_metrics(g):
    e=pd.to_numeric(g['error_C'],errors='coerce').dropna().to_numpy(float)
    return pd.Series({'n_rows':len(e),'MAE_C':float(np.mean(np.abs(e))) if len(e) else np.nan,'RMSE_C':float(np.sqrt(np.mean(e**2))) if len(e) else np.nan,'Bias_C':float(np.mean(e)) if len(e) else np.nan})

TEST_ALL=PRED_LONG[PRED_LONG['split'].eq('test')].copy()
TEST_INSIDE=TEST_ALL[TEST_ALL['target_context'].eq('inside_event')].copy()

# Compare identical test temperature targets with valid predictions from both models.
def exact_common_rows(frame,methods=('Room-specific Extra Trees','Unified derivative NN')):
    wide=frame.pivot_table(index='_row_id',columns='method',values='pred_T',aggfunc='first')
    for m in methods:
        if m not in wide.columns: wide[m]=np.nan
    return set(wide.dropna(subset=list(methods)).index.astype(int))

common_inside=exact_common_rows(TEST_INSIDE); MAIN_COMMON=TEST_INSIDE[TEST_INSIDE['_row_id'].isin(common_inside)].copy()
main_metrics=MAIN_COMMON.groupby(['phase','method'],as_index=False).apply(temperature_metrics,include_groups=False).reset_index(drop=True)
main_metrics.to_csv(TABLE_DIR/'MAIN_test_inside_event_temperature_metrics_COMMON_SUPPORT.csv',index=False)

NN_FULL=TEST_INSIDE[TEST_INSIDE['method'].eq('Unified derivative NN')].copy()
nn_full_metrics=NN_FULL.groupby(['phase','method'],as_index=False).apply(temperature_metrics,include_groups=False).reset_index(drop=True)
nn_full_metrics.to_csv(TABLE_DIR/'UnifiedNN_test_inside_event_temperature_metrics_FULL_SUPPORT.csv',index=False)

# Horizon-specific temperature diagnostics include valid inside-event and permitted
# post-end targets.
common_all=exact_common_rows(TEST_ALL); HCOMMON=TEST_ALL[TEST_ALL['_row_id'].isin(common_all)].copy()
horizon_metrics=HCOMMON.groupby(['phase','method','horizon_min'],as_index=False).apply(temperature_metrics,include_groups=False).reset_index(drop=True)
horizon_metrics.to_csv(TABLE_DIR/'test_temperature_metrics_by_horizon_ALL_CONTEXT_COMMON_SUPPORT.csv',index=False)
support=(HCOMMON.drop_duplicates(['_row_id','phase','horizon_min']).groupby(['phase','horizon_min'],as_index=False).size().rename(columns={'size':'n_common_rows'}))
support.to_csv(TABLE_DIR/'test_temperature_horizon_common_support_counts.csv',index=False)

print('PRIMARY FAIR TEMPERATURE COMPARISON — identical chronological TEST split + exact common prediction rows')
display(main_metrics.round(4))
print('\nUNIFIED NN — full inside-event test support'); display(nn_full_metrics.round(4))

fig,axes=plt.subplots(1,2,figsize=(15,5.5),constrained_layout=True)
for ax,phase in zip(axes,PHASES):
    d=horizon_metrics[horizon_metrics['phase'].eq(phase)]
    for method,g in d.groupby('method'):
        g=g.sort_values('horizon_min'); ax.plot(g['horizon_min'],g['MAE_C'],marker='o',markersize=3,label=method)
    ax.set_xlim(0,PAPER_TEMPERATURE_XMAX_BY_PHASE[phase])
    ax.set(title=f'{phase.capitalize()} — all exported target contexts, exact common support',xlabel='Forecast horizon (min)',ylabel='MAE (°C)')
    ax.grid(alpha=.25)
    ax2=ax.twinx(); s=support[support['phase'].eq(phase)].sort_values('horizon_min')
    ax2.plot(s['horizon_min'],s['n_common_rows'],linestyle='--',alpha=.35,label='Common test rows n')
    ax2.set_ylabel('Common test rows n'); ax.legend(frameon=False,loc='upper left'); ax2.legend(frameon=False,loc='upper right')
fig.savefig(FIG_DIR/'PAPER_temperature_MAE_by_horizon_ALL_CONTEXT_COMMON_SUPPORT.png',dpi=300,bbox_inches='tight'); plt.show()

# Directed-rate diagnostics.
rate_test=DATA[DATA['split'].eq('test')&DATA['use_for_temperature_training']].copy()
rate_test['rate_error_C_per_min']=pd.to_numeric(rate_test['pred_directed_rate_UnifiedNN'],errors='coerce')-pd.to_numeric(rate_test['target_directed_rate_C_per_min'],errors='coerce')
rate_rows=[]
for phase,g in rate_test.groupby('phase'):
    e=pd.to_numeric(g['rate_error_C_per_min'],errors='coerce').dropna().to_numpy(float)
    rate_rows.append({'phase':phase,'n_rows':len(e),'MAE_C_per_min':float(np.mean(np.abs(e))) if len(e) else np.nan,'RMSE_C_per_min':float(np.sqrt(np.mean(e**2))) if len(e) else np.nan,'Bias_C_per_min':float(np.mean(e)) if len(e) else np.nan})
pd.DataFrame(rate_rows).to_csv(TABLE_DIR/'UnifiedNN_test_directed_rate_metrics_FULL_SUPPORT.csv',index=False)


In [ ]:

# Reference training manifest

best_ckpt=safe_torch_load(MODEL_DIR/'unified_directed_rate_nn.pt',map_location='cpu')
manifest={
    'notebook':'02_train_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL_GPU',
    'data_dir':str(DATA_DIR),'output_dir':str(OUT_ROOT),'shared_split_map':str(SPLIT_MAP_PATH),'shared_split_sha256':SHARED_SPLIT_SHA256,
    'same_chronological_split_for_all_models':True,'authoritative_split_modified_for_baselines':False,
    'temperature_supervision':'LONG exact-horizon trajectories from Notebook 01','duration_supervision':'authoritative duration anchors from Notebook 01 including event_start_asof',
    'unified_nn':{
        'scope':'one network across all rooms and both phases','room_identity_input':False,'output':'directed rate v = thermal_mode_sign * dT/dt',
        'temperature_reconstruction':'trapezoidal integration from measured current T','separate_duration_head':False,'slope_loss':False,
        'trajectory_loss':'0.5*(TRAIN-normalized rate MSE + TRAIN-normalized reconstructed-T MSE)',
        'duration_loss':'q-weighted TRAIN-normalized authoritative-anchor remaining-time MSE','normalization':'TRAIN-only phase-specific','phase_balance':'equal heating/cooling',
        'scheduler':'ReduceLROnPlateau on balanced unweighted normalized validation duration MSE','best_epoch':best_ckpt.get('best_epoch'),
        'best_validation_duration_norm_mse':best_ckpt.get('best_validation_duration_norm_mse'),'test_used_for_model_selection':False,
        'resume_checkpoint':str(CHECKPOINT_DIR/'resume_state.pt'),'mixed_precision':USE_AMP,
    },
    'extra_trees':{'scope':'room/phase specific only with sufficient TRAIN support from the same authoritative split','unsupported_policy':'NaN; never borrow validation/test','fair_comparison':'exact common prediction rows/anchors'},
    'model_inputs':MODEL_INPUTS,'loss_scales':LOSS_SCALES,
    'artifacts':{'selected_nn':str(MODEL_DIR/'unified_directed_rate_nn.pt'),'preprocessor':str(MODEL_DIR/'unified_numeric_preprocessor.joblib'),'et_root':str(ET_ROOT)},
}
for name in ['training_manifest_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL.json','training_manifest_UNIFIED_DERIVATIVE_DURATION_AWARE_FINAL.json','training_manifest_UNIFIED_DERIVATIVE_DURATION_AWARE.json']:
    with open(OUT_ROOT/name,'w') as f: json.dump(manifest,f,indent=2,default=str)
print('Notebook 02 PAPER FINAL complete.')
print('Training run:',OUT_ROOT)
print('Best validation epoch:',best_ckpt.get('best_epoch'))
print('Shared split SHA256:',SHARED_SPLIT_SHA256)
